# TrekGuardian-v1: Hypoxia Analysis Notebook

This notebook demonstrates how to work with the 3 hypoxia-related datasets from TrekGuardian-v1.

## Datasets Included:
1. Respiratory and Pulse Oximetry Waveforms (2.5 GB)
2. VitalDB - Multi-Parameter Vital Signs (95.4 GB)
3. Temporal Respiratory Support in ICU (18 GB)

## Setup and Imports

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# PhysioNet libraries
import wfdb
import vitaldb

# Configure matplotlib
%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Configure paths
DATASETS_PATH = Path('.')
print(f"Datasets path: {DATASETS_PATH}")
print(f"Available directories: {list(DATASETS_PATH.glob('*_*'))}")

## Dataset 1: Respiratory and Pulse Oximetry Analysis

In [ ]:
# Load respiratory oximetry dataset
dataset1_path = DATASETS_PATH / '1_respiratory-oximetry'

if dataset1_path.exists():
    # List available records
    records = list(dataset1_path.glob('*.dat'))
    print(f"Found {len(records)} records in respiratory-oximetry dataset")
    print(f"First record: {records[0].name if records else 'None'}")
else:
    print(f"Dataset 1 not found at {dataset1_path}")
    print("Please download the dataset first using download_datasets.sh")

In [ ]:
# Load and analyze a respiratory oximetry record
try:
    record_path = str(dataset1_path / 'a001')
    record = wfdb.rdrecord(record_path)
    
    print(f"Record: {record.recordname}")
    print(f"Number of samples: {record.sig_len}")
    print(f"Sample rate: {record.fs} Hz")
    print(f"Signal names: {record.sig_name}")
    print(f"Duration: {record.sig_len / record.fs:.2f} seconds")
except Exception as e:
    print(f"Could not load record: {e}")

In [ ]:
# Extract and analyze SpO2 signal
if 'record' in locals() and 'SpO2' in record.sig_name:
    spo2_idx = record.sig_name.index('SpO2')
    spo2_data = record.p_signal[:, spo2_idx]
    time_axis = np.arange(len(spo2_data)) / record.fs
    
    # Calculate statistics
    print(f"SpO2 Statistics:")
    print(f"  Mean: {np.mean(spo2_data):.2f}%")
    print(f"  Min: {np.min(spo2_data):.2f}%")
    print(f"  Max: {np.max(spo2_data):.2f}%")
    print(f"  Std Dev: {np.std(spo2_data):.2f}%")
    
    # Detect hypoxia events (SpO2 < 90%)
    hypoxia_threshold = 90
    hypoxia_events = spo2_data < hypoxia_threshold
    hypoxia_count = np.sum(hypoxia_events)
    print(f"\nHypoxia events (SpO2 < {hypoxia_threshold}%): {hypoxia_count} samples")

In [ ]:
# Plot SpO2 waveform
if 'spo2_data' in locals():
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(time_axis, spo2_data, linewidth=1.5, label='SpO2')
    ax.axhline(y=90, color='r', linestyle='--', label='Hypoxia Threshold')
    ax.fill_between(time_axis, 90, spo2_data, where=(spo2_data < 90), 
                     alpha=0.3, color='red', label='Hypoxia Zone')
    ax.set_xlabel('Time (seconds)')
    ax.set_ylabel('SpO2 (%)')
    ax.set_title('Respiratory Oximetry: SpO2 During Simulated Apnea')
    ax.set_ylim([70, 105])
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## Dataset 2: VitalDB Analysis

In [ ]:
# Load VitalDB data
try:
    # Load SpO2 data from first case
    caseid = 1
    spo2_data = vitaldb.load_case(caseid=caseid, tnames='Solar8000/SpO2', interval=1)
    
    print(f"Case ID: {caseid}")
    print(f"SpO2 data shape: {spo2_data[0].shape}")
    print(f"Duration: {len(spo2_data[0])} seconds ({len(spo2_data[0])/60:.1f} minutes)")
    print(f"\nSpO2 Statistics:")
    print(f"  Mean: {np.mean(spo2_data[0]):.2f}%")
    print(f"  Min: {np.min(spo2_data[0]):.2f}%")
    print(f"  Max: {np.max(spo2_data[0]):.2f}%")
except Exception as e:
    print(f"Could not load VitalDB data: {e}")
    print("Make sure VitalDB is downloaded and installed correctly.")

In [ ]:
# Load multiple parameters from VitalDB
try:
    # Load SpO2, Heart Rate, and Temperature
    params = ['Solar8000/SpO2', 'Solar8000/HR', 'Solar8000/BT']
    data = vitaldb.load_case(caseid=1, tnames=params, interval=10)  # 10-second intervals for efficiency
    
    # Create DataFrame
    df = pd.DataFrame({
        'SpO2': data[0],
        'HR': data[1],
        'Temperature': data[2]
    })
    
    print("Vital Signs Correlation:")
    print(df.corr())
    print("\nBasic Statistics:")
    print(df.describe())
except Exception as e:
    print(f"Could not load multiple parameters: {e}")

## Dataset 3: Temporal Respiratory Support Analysis

In [ ]:
# Load temporal respiratory support data
dataset3_path = DATASETS_PATH / '3_temporal-respiratory'

if dataset3_path.exists():
    csv_files = list(dataset3_path.glob('*.csv'))
    print(f"Found {len(csv_files)} CSV files")
    for f in csv_files[:5]:  # Show first 5
        print(f"  - {f.name}")
    
    # Try to load main data file
    if csv_files:
        df = pd.read_csv(csv_files[0])
        print(f"\nData shape: {df.shape}")
        print(f"Columns: {df.columns.tolist()[:10]}...")  # First 10 columns
        print(f"\nFirst few rows:")
        print(df.head())
else:
    print(f"Dataset 3 not found at {dataset3_path}")

## Comparative Analysis

In [ ]:
# Create comparison table
comparison = pd.DataFrame({
    'Dataset': [
        'Respiratory Oximetry',
        'VitalDB',
        'Temporal Respiratory'
    ],
    'Size (GB)': [2.5, 95.4, 18.0],
    'Subjects': [20, 6388, 50920],
    'Primary Focus': [
        'SpO2 during apnea',
        'Multi-param vital signs',
        'ICU respiratory trends'
    ],
    'Duration': [
        '10-30 min',
        '2-6 hours',
        '90 days'
    ],
    'Hypoxia Events': [
        'Simulated',
        'Clinical',
        'Natural ICU'
    ]
})

print("\n" + "="*80)
print("TrekGuardian-v1 Datasets Comparison")
print("="*80)
print(comparison.to_string(index=False))
print("="*80)

## Hypoxia Analysis Pipeline

Complete workflow for hypoxia detection and analysis

In [ ]:
def analyze_hypoxia(spo2_data, fs=1, threshold=90, duration_threshold=30):
    """
    Analyze SpO2 data for hypoxia events
    
    Parameters:
    -----------
    spo2_data : array
        SpO2 values
    fs : float
        Sampling frequency (Hz)
    threshold : float
        SpO2 threshold for hypoxia (default 90%)
    duration_threshold : float
        Minimum event duration in seconds
    """
    
    # Detect hypoxia events
    hypoxia = spo2_data < threshold
    
    # Find event boundaries
    diff = np.diff(hypoxia.astype(int))
    starts = np.where(diff == 1)[0] + 1
    ends = np.where(diff == -1)[0] + 1
    
    # Calculate event durations
    events = []
    for start, end in zip(starts, ends):
        duration = (end - start) / fs
        if duration >= duration_threshold:
            events.append({
                'start': start,
                'end': end,
                'duration': duration,
                'min_spo2': np.min(spo2_data[start:end]),
                'mean_spo2': np.mean(spo2_data[start:end])
            })
    
    return {
        'total_events': len(events),
        'total_hypoxia_time': np.sum(hypoxia) / fs,
        'mean_spo2': np.mean(spo2_data),
        'min_spo2': np.min(spo2_data),
        'events': events
    }

# Example usage
if 'spo2_data' in locals():
    results = analyze_hypoxia(spo2_data, fs=1, threshold=90)
    print("Hypoxia Analysis Results:")
    print(f"  Total events: {results['total_events']}")
    print(f"  Total hypoxia time: {results['total_hypoxia_time']:.1f} seconds")
    print(f"  Mean SpO2: {results['mean_spo2']:.2f}%")
    print(f"  Minimum SpO2: {results['min_spo2']:.2f}%")

## References and Resources

- PhysioNet: https://physionet.org/
- WFDB Library: https://wfdb.io/
- VitalDB Python Package: https://pypi.org/project/vitaldb/
- Hypoxia Research: See DATASET_GUIDE.md